In [ ]:
import dataset
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

seed = 123

df = dataset.read_agg(month_start=1, month_end=12)
df = df[df['pickup_year'] == 2024]
df.info()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# Recast object types -> category
# Not sure why this changes from typecasting in `dataset.clean()`
df = df.astype({
    'vendor': 'category',
    # 'ratecode': 'category',
    'pickup_zone': 'category',
    'dropoff_zone': 'category',
    'route': 'category',
})

# Remove waste columns
df = df.drop(columns=[
    'total_passenger_count',  
    'avg_passenger_count',    
    'total_trip_distance',    
    'avg_trip_distance',      
    'total_fare_amount',      
    'avg_fare_amount',        
    'total_extra',            
    'avg_extra',              
    'total_mta_tax',          
    'avg_mta_tax',            
    'total_tip_amount',       
    'avg_tip_amount',         
    'total_tolls_amount',     
    'avg_tolls_amount',       
    'total_impr_surcharge',
    'avg_impr_surcharge',     
    'total_revenue',          
    'avg_revenue',            
    'total_airport_fee',      
    'avg_airport_fee',       
])

# Onehot encode the following:
#
#  4   pickup_service_zone    category
#  5   pickup_zone            category  
#  6   dropoff_service_zone   category
#  7   dropoff_zone           category  
#  8   route                  category  
#  9   service_route          category  
#  10  vendor                 category  
#  11  ratecode               category  [NOT USED]
#  12  payment_type           category  [NOT USED]
#
# These columns only have a few categories
#
# => onehot encoding is okay
columns_to_onehot_encode = [
    'pickup_dow',
    'pickup_service_zone',
    'dropoff_service_zone',
    'service_route',
    'vendor',
    'pickup_dow',
    # 'ratecode',
    # 'payment_type',
]

# These columns have LOADS of categories
#
# => ordinal encoding is needed             <---- not doing this gives 500GB encoded dataset
#``
# (tried TargetEncoder, but still ended up at 22GB - sticking to ordinal)
columns_to_ordinal_encode = [
    'pickup_zone',
    'dropoff_zone',
    'route'
]

# Split `y` BEFORE pipeline
y = df['total_ride_count']
X = df.drop(columns=['total_ride_count'])

# Pipeline
#
# Transfroms the categorical columns to onehot-encoded types
pipeline = ColumnTransformer(
    transformers=[
        #                               +--- `sparse_output=True` stops the encoder duplicating data
        #                               |
        ("onehot", OneHotEncoder(sparse_output=True), columns_to_onehot_encode), 
        ("ordinal", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_ordinal_encode),
    ],
    remainder='passthrough'
)
# pipeline.set_output(transform='pandas') # DEPRECATED: returning pandas cost too much memory
X_encoded = pipeline.fit_transform(X)

# Train vs Test
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=seed)

# Train vs Validation
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.3, random_state=seed)

(         pickup_year  pickup_month  pickup_week  ...   Low  Precip Snow
 2562199         2024            11           47  ...  53.0    0.00  0.0
 556690          2024             3           11  ...  51.0    0.00  0.0
 250357          2024             2            5  ...  31.0    0.00  0.0
 2190616         2024            10           40  ...  55.0    0.00  0.0
 2705815         2024            12           49  ...  31.0    0.00  0.0
 ...              ...           ...          ...  ...   ...     ...  ...
 1954939         2024             9           36  ...  59.0    0.24  0.0
 1241071         2024             6           23  ...  66.0    0.00  0.0
 28040           2024             1            1  ...  28.0    0.00  0.0
 277881          2024             2            6  ...  33.0    0.00  0.0
 773646          2024             4           15  ...  51.0    0.00  0.0
 
 [2310246 rows x 16 columns],
 2562199     18
 556690     126
 250357      19
 2190616      2
 2705815      1
           .

In [ ]:
xg = XGBRegressor(
    tree_method="hist",
    eval_metric=mean_absolute_error,
)
xg.fit(X_train, y_train)

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:pickup_service_zone: category, pickup_zone: category, dropoff_service_zone: category, dropoff_zone: category, route: category, service_route: category, vendor: category

In [ ]:
import helpers

training_predictions = xg.predict(X_train)
helpers.analysis(y_train, X_train)